In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#Loading the data
train = pd.read_csv("/kaggle/input/titanic/train.csv")
test = pd.read_csv("/kaggle/input/titanic/test.csv")

train.head()

In [ ]:
#check for data types and null count
train.info()
test.info()

In [ ]:
train.describe()

In [ ]:
#seperating the varibale to their respective testing and training set.
y_train = train[['Survived']]
x_train = train.copy()

x_test = test.copy()

In [ ]:
def NAN (df):
    for col in (df.columns):
        if df[col].isna().astype(int).sum() > 0:
            print(f'Missing values in {col}:',df[col].isna().astype(int).sum())
            print(f'Missing percentage {col}',(df[col].isna().astype(int).sum())/x_train.shape[0])

NAN(x_train)

In [ ]:
#try to estimate value for age and embarked.
#we can drop name(no relevance in survival), passengerid(just an identifier) and cabin(too many missing values)
# Concatenate the original DataFrame and the dummy variables
from sklearn.preprocessing import StandardScaler

def pre (df):
    mean_age = df['Age'].mean()
    mean_age = np.round(mean_age,2)
    df['Age'] = df['Age'].replace(np.nan,mean_age)
    df['Sex'] = df['Sex'].map({'female':0,'male':1})
    df.drop(['Name', 'PassengerId','Cabin','Ticket'],axis=1,inplace=True)
    dummy1 = pd.get_dummies(df['Embarked']).astype(int)
    dummy1.rename(columns={'C':'Embarked_c','Q':'Embarked_q','S':'Embarked_s'},inplace=True)
    df = pd.concat([df, dummy1], axis=1)
    df.drop('Embarked',inplace=True,axis=1)
    scale = StandardScaler()
    col = df.columns
    df_scale = scale.fit_transform(df)
    df_scale = pd.DataFrame(df_scale,columns=col)
    return df


    


In [ ]:
x_train = pre(x_train)
x_train

In [ ]:
x_test = pre(x_test)


In [ ]:
NAN(x_test)
x_test['Fare'] = x_test['Fare'].replace(np.nan,x_test['Fare'].mean())

In [ ]:
import seaborn as sns
sns.heatmap(x_train.corr(),annot=True)

In [ ]:
sns.pairplot(data=x_train)

In [ ]:
#checking for class imbalance
x_train['Survived'].value_counts()

In [ ]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()
lr.fit(x_train.drop(['Survived','Embarked_q'],axis=1),y_train)
print('R2 score:',lr.score(x_train.drop(['Survived','Embarked_q'],axis=1),y_train))
y_pred = lr.predict(x_test.drop(['Embarked_q'],axis=1))
y_pred


In [ ]:
df = pd.DataFrame({'Survived':y_pred})
df = pd.concat([test['PassengerId'],df],axis=1)


In [ ]:
df.to_csv('Survived.csv',index=False)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
rf.fit(x_train.drop(['Survived','Embarked_q'],axis=1),y_train)
print('R2 score',rf.score(x_train.drop(['Survived','Embarked_q'],axis=1),y_train))
yrf_pred = rf.predict(x_test.drop(['Embarked_q'],axis=1))
yrf_pred

In [ ]:
df1 = pd.DataFrame({'Survived':yrf_pred})
df1 = pd.concat([test['PassengerId'],df1],axis=1)
df1.to_csv('Survived(Random Forrest Classifier).csv',index=False)

In [ ]:
from sklearn.svm import SVC
svc = SVC()
svc.fit(x_train.drop(['Survived','Embarked_q'],axis=1),y_train)
print('R2 score',svc.score(x_train.drop(['Survived','Embarked_q'],axis=1),y_train))
ysvc_pred = rf.predict(x_test.drop(['Embarked_q'],axis=1))
ysvc_pred

In [ ]:
df2 = pd.DataFrame({'Survived':ysvc_pred})
df2 = pd.concat([test['PassengerId'],df2],axis=1)
df2.to_csv('Survived(SVM).csv',index=False)